In [ ]:
# 安裝必要套件
!pip install yfinance pandas matplotlib numpy -q
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


# 第 8 堂：從回測到實盤 + 期末專案提案


## 期末專案：量化策略提案

請在下方填入你的策略想法，完成後在 GitHub `projects/` 目錄提交。

---

**1. 策略名稱：** 

**2. 市場標的：** 

**3. 進場邏輯：** 

**4. 停損規則：** 

**5. 資金管理原則：** 

**6. 預期資料來源：** 

**7. 初步回測時間框架：** 


In [ ]:
# Walk-Forward 概念展示
TICKER = "2330.TW"
df = yf.Ticker(TICKER).history(period="5y")[['Close']].copy()
train = df[:'2022']
validate = df['2023']
test = df['2024':]
def simple_ma_backtest(data, fast=5, slow=20):
    d = data.copy()
    d['MA_fast'] = d['Close'].rolling(fast).mean()
    d['MA_slow'] = d['Close'].rolling(slow).mean()
    d['signal'] = (d['MA_fast'] > d['MA_slow']).astype(int)
    d['ret'] = d['Close'].pct_change()
    d['strat'] = d['ret'] * d['signal'].shift(1)
    strat_ret = d['strat'].dropna()
    total = (1 + strat_ret).prod() - 1
    sharpe = strat_ret.mean() / strat_ret.std() * np.sqrt(252) if strat_ret.std() > 0 else 0
    return total, sharpe
print(f"訓練集：{len(train)} 筆 | 驗證集：{len(validate)} 筆 | 測試集：{len(test)} 筆")
for fast, slow in [(5, 20), (10, 60), (5, 60)]:
    tr_ret, tr_sh = simple_ma_backtest(train, fast, slow)
    va_ret, va_sh = simple_ma_backtest(validate, fast, slow)
    te_ret, te_sh = simple_ma_backtest(test, fast, slow)
    print(f"MA{fast}/{slow} — Train:{tr_ret:.2%}(sh={tr_sh:.2f}) Valid:{va_ret:.2%}(sh={va_sh:.2f}) Test:{te_ret:.2%}(sh={te_sh:.2f})")
